# Retrain YOLO11 for Face Recognition

This notebook fine-tunes YOLO11 on a custom dataset with two classes:

| Class ID | Label | Source |
|----------|-------|--------|
| 0 | `adnan` | Your selfie photos in `my_photos/` |
| 1 | `unknown_face` | Public faces from the [colleague photos + HuggingFace portraits](http://shuoyang1213.me/WIDERFACE/) dataset |

**Workflow:**
1. Your selfie photos are pre-loaded in the `my_photos/` folder (no manual annotation needed).
2. We download a subset of the colleague photos + HuggingFace portraits dataset for the "unknown_face" class.
3. A pre-trained YOLO11-face detector auto-annotates every image with bounding boxes.
4. We train YOLO11m on this two-class dataset. YOLO11's built-in augmentation (mosaic, mixup, flip, rotation) compensates for the small dataset size.
5. The trained model is exported to ONNX for cross-platform deployment.

In [ ]:
!pip install --no-cache-dir -q -r requirements.txt

import os
import shutil
import random
import zipfile
from pathlib import Path
from ultralytics import YOLO
from huggingface_hub import hf_hub_download, snapshot_download

## Step 1: Verify your photos

Your selfie photos should already be in the `my_photos/` folder (uploaded by `deploy.sh` or `upload-to-workbench.sh`).

- Supported formats: `.jpg`, `.jpeg`, `.png`
- More variety (lighting, angles, backgrounds) improves training accuracy
- One face per photo works best

In [ ]:
MY_PHOTOS_DIR = Path("my_photos")
if not MY_PHOTOS_DIR.exists() or len(list(MY_PHOTOS_DIR.glob("*.jpg")) + list(MY_PHOTOS_DIR.glob("*.jpeg")) + list(MY_PHOTOS_DIR.glob("*.png"))) == 0:
    print("WARNING: No photos found in my_photos/ directory!")
    print("Please place ~50 selfie photos there before continuing.")
    print("Supported formats: .jpg, .jpeg, .png")
else:
    photo_count = len(list(MY_PHOTOS_DIR.glob("*.jpg")) + list(MY_PHOTOS_DIR.glob("*.jpeg")) + list(MY_PHOTOS_DIR.glob("*.png")))
    print(f"Found {photo_count} photos in my_photos/")

## Step 2: Load faces for the 'unknown_face' class

We combine two sources for the strongest possible unknown class:
1. **Real colleague photos** from `unknown_face/` — same events, lighting, and camera style as your selfies
2. **HuggingFace portraits portraits** from [Labeled Faces in the Wild](http://vis-www.cs.umass.edu/lfw/) — diverse close-up faces for broader generalization

In [ ]:
!pip install -q datasets

from datasets import load_dataset
from PIL import Image

UNKNOWN_DIR = Path("unknown_face")
UNKNOWN_COMBINED_DIR = Path("unknown_combined")
NUM_PORTRAITS = 200

# --- Source 1: Real colleague photos ---
custom_photos = []
if UNKNOWN_DIR.exists():
    custom_photos = sorted(UNKNOWN_DIR.glob("*.jpg")) + sorted(UNKNOWN_DIR.glob("*.jpeg")) + sorted(UNKNOWN_DIR.glob("*.png"))
print(f"Real colleague photos: {len(custom_photos)}")

# --- Source 2: HuggingFace Realistic Portraits (high-quality close-ups) ---
print(f"Downloading {NUM_PORTRAITS} realistic face portraits from HuggingFace...")
ds = load_dataset("prithivMLmods/Realistic-Face-Portrait-1024px", split="train", streaming=True)
portrait_count = 0
portrait_dir = Path("/tmp/portraits_cache")
portrait_dir.mkdir(exist_ok=True)
for example in ds:
    if portrait_count >= NUM_PORTRAITS:
        break
    try:
        img = example["image"]
        if img.width >= 256:
            img = img.resize((512, 512))
            img.save(portrait_dir / f"port_{portrait_count:04d}.jpg", quality=95)
            portrait_count += 1
    except:
        continue
print(f"Downloaded {portrait_count} portrait faces")

# --- Combine all unknowns into staging directory ---
if UNKNOWN_COMBINED_DIR.exists():
    shutil.rmtree(UNKNOWN_COMBINED_DIR)
UNKNOWN_COMBINED_DIR.mkdir(exist_ok=True)

idx = 0
for p in custom_photos:
    shutil.copy2(p, UNKNOWN_COMBINED_DIR / f"unknown_{idx:04d}.jpg")
    idx += 1

for p in sorted(portrait_dir.glob("*.jpg")):
    shutil.copy2(p, UNKNOWN_COMBINED_DIR / f"unknown_{idx:04d}.jpg")
    idx += 1

print(f"Combined unknown_face class: {idx} photos")
print(f"  - {len(custom_photos)} real colleagues")
print(f"  - {portrait_count} HuggingFace portraits")


## Step 3: Auto-annotate all images

Instead of manually drawing bounding boxes, we use a pre-trained [YOLO11-face](https://huggingface.co/AdamCodd/YOLOv11n-face-detection) model to detect faces in every image. For each detected face we write a YOLO-format label file (`class_id cx cy w h`) — then assign:

- **class 0** (`adnan`) to detections in your selfie photos
- **class 1** (`unknown_face`) to detections in the HuggingFace portraits photos

The dataset is split 80/20 into train and validation sets.

In [ ]:
DATASET_DIR = Path("face-dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ["train", "val"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True)

detector_path = hf_hub_download(repo_id="AdamCodd/YOLOv11n-face-detection", filename="model.pt")
detector = YOLO(detector_path)


def auto_annotate(image_path, class_id, output_img_dir, output_lbl_dir, prefix="img"):
    """Detect face, save image and YOLO-format label."""
    try:
        results = detector.predict(str(image_path), verbose=False, conf=0.3)
    except Exception:
        return False
    if len(results[0].boxes) == 0:
        return False

    img_name = f"{prefix}_{image_path.stem}.jpg"
    lbl_name = f"{prefix}_{image_path.stem}.txt"

    shutil.copy2(image_path, output_img_dir / img_name)

    img_h, img_w = results[0].orig_shape
    with open(output_lbl_dir / lbl_name, "w") as f:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = ((x1 + x2) / 2) / img_w
            cy = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            f.write(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
    return True


user_photos = sorted(MY_PHOTOS_DIR.glob("*.jpg")) + sorted(MY_PHOTOS_DIR.glob("*.jpeg")) + sorted(MY_PHOTOS_DIR.glob("*.png"))
unknown_photos = sorted(UNKNOWN_COMBINED_DIR.glob("*.jpg"))

random.shuffle(user_photos)
random.shuffle(unknown_photos)

split_idx_user = int(len(user_photos) * 0.8)
split_idx_unknown = int(len(unknown_photos) * 0.8)

annotated = {"train": 0, "val": 0}
for photos, class_id, prefix in [
    (user_photos, 0, "adnan"),
    (unknown_photos, 1, "unknown"),
]:
    split_idx = int(len(photos) * 0.8)
    for split, photo_list in [("train", photos[:split_idx]), ("val", photos[split_idx:])]:
        for p in photo_list:
            if auto_annotate(p, class_id, DATASET_DIR / "images" / split, DATASET_DIR / "labels" / split, prefix):
                annotated[split] += 1

print(f"Dataset created: {annotated['train']} train, {annotated['val']} val images")

In [ ]:
data_yaml = DATASET_DIR / "data.yaml"
data_yaml.write_text(f"""path: {DATASET_DIR.resolve()}
train: images/train
val: images/val

nc: 2
names:
  0: adnan
  1: unknown_face
""")
print(f"Created {data_yaml}")
print(data_yaml.read_text())

## Step 4: Train the model

We fine-tune `yolo11m.pt` (the medium variant — 20.4M params, 7.7x more capacity than nano for learning subtle facial differences) on our two-class dataset.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `epochs` | 100 | Deep convergence with large dataset (~1K images) |
| `imgsz` | 640 | Standard YOLO input resolution |
| `batch` | 16 | Larger batch for stable gradients on GPU (L4 has headroom) |
| `device` | 0 | GPU training (falls back to CPU if unavailable) |
| `workers` | 0 | Required in containers (small /dev/shm) |
| `mosaic` | 1.0 | Combines 4 images — creates group-photo-like scenes |
| `close_mosaic` | 20 | Keep mosaic on longer to reduce transition oscillation |
| `mixup` | 0.0 | Disabled — blending faces confuses identity discrimination |
| `fliplr` | 0.5 | Horizontal flip doubles effective dataset size |
| `patience` | 15 | Tolerates metric oscillation without premature stopping |


In [ ]:
model = YOLO("yolo11m.pt")

results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    mosaic=1.0,
    close_mosaic=20,
    mixup=0.0,
    fliplr=0.5,
    patience=15,
    project="runs/face-recognition",
    name="train",
    exist_ok=True,
)


In [ ]:
# ============================================================
# EXPERIMENT MODE: Uncomment ONE config block to test
# Run each experiment separately, compare final mAP50/mAP50-95
# ============================================================

# --- Experiment A: Optimized (committed config) ---
# Best for: smoother convergence, no face blending
experiment = {
    "name": "A_optimized",
    "batch": 16, "close_mosaic": 20, "mixup": 0.0,
    "patience": 15, "freeze": None,
}

# --- Experiment B: Larger batch ---
# Best for: testing if 32 batch is too few steps
# experiment = {
#     "name": "B_batch32",
#     "batch": 32, "close_mosaic": 20, "mixup": 0.0,
#     "patience": 15, "freeze": None,
# }

# --- Experiment C: Extended mosaic ---
# Best for: more augmentation variety in early training
# experiment = {
#     "name": "C_mosaic30",
#     "batch": 16, "close_mosaic": 30, "mixup": 0.0,
#     "patience": 15, "freeze": None,
# }

# --- Experiment D: Frozen backbone ---
# Best for: preventing overfitting, faster training
# experiment = {
#     "name": "D_freeze10",
#     "batch": 16, "close_mosaic": 20, "mixup": 0.0,
#     "patience": 15, "freeze": 10,
# }

print(f"Running experiment: {experiment["name"]}")
model = YOLO("yolo11m.pt")

results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=experiment["batch"],
    device=0,
    workers=0,
    mosaic=1.0,
    close_mosaic=experiment["close_mosaic"],
    mixup=experiment["mixup"],
    fliplr=0.5,
    patience=experiment["patience"],
    freeze=experiment["freeze"],
    project="runs/face-recognition",
    name=experiment["name"],
    exist_ok=True,
)

# Print summary for comparison
print(f"
=== {experiment["name"]} RESULTS ===")
print(f"Best mAP50: {results.results_dict["metrics/mAP50(B)"]:.4f}")
print(f"Best mAP50-95: {results.results_dict["metrics/mAP50-95(B)"]:.4f}")
print(f"Precision: {results.results_dict["metrics/precision(B)"]:.4f}")
print(f"Recall: {results.results_dict["metrics/recall(B)"]:.4f}")


## Step 5: Export to ONNX

ONNX (Open Neural Network Exchange) is an open format that enables cross-platform inference. Exporting to ONNX lets us deploy the model on OpenVINO, TensorRT, or any ONNX-compatible runtime without depending on PyTorch at serving time.

In [ ]:
best_model_path = list(Path("runs").rglob("face-recognition/train/weights/best.pt"))
if best_model_path:
    best_pt = best_model_path[0]
    print(f"Found: {best_pt}")
    trained_model = YOLO(str(best_pt))
    onnx_path = trained_model.export(format="onnx")
    print(f"ONNX model exported to: {onnx_path}")
else:
    print("Training weights not found. Ensure training completed successfully.")

## Done!

The retrained model can now classify detected faces as **adnan** (class 0) or **unknown_face** (class 1).

**Outputs:**
- PyTorch weights: `runs/face-recognition/train/weights/best.pt`
- ONNX model: `runs/face-recognition/train/weights/best.onnx`

**Next:** Open **`03-test-retrained-model.ipynb`** to test the retrained model on new images and video.